In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


In [ ]:
# The following two lines are not needed if you have topologicpy already pip installed.
import sys
sys.path.append("C:/Users/wassimj/Documents/GitHub/")

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.TGraph import TGraph
from topologicpy.Plotly import Plotly
from topologicpy.Color import Color
from topologicpy.Vector import Vector
from topologicpy.Helper import Helper
from topologicpy.DGL import DGL
import math
print("Done importing libraries")

In [ ]:
cc = CellComplex.Prism(width=10, length=10, height=30, wSides=10, placement="bottom")
Topology.Show(cc)

In [ ]:
cells = CellComplex.Cells(cc)
selectors = []
for cell in cells:
    centroid = Topology.Centroid(_as_topology(cell))
    z = Vertex.Z(_as_topology(centroid))
    if z < 6:
        cat = 0
    elif z < 12:
        cat = 1
    elif z < 18:
        cat = 2
    elif z < 24:
        cat = 3
    else:
        cat = 4
    d = Dictionary.ByKeyValue("id", cat)
    centroid = Topology.SetDictionary(centroid, d)
    selectors.append(centroid)

cc = Topology.TransferDictionariesBySelectors(cc, selectors, tranCells=True)

In [ ]:
g = TGraph.ByTopology(cc)
g_vertices = TGraph.Vertices(g)
for g_vertex in g_vertices:
    d = _record_dictionary(g_vertex)
    print(Dictionary.Keys(d), Dictionary.Values(d))

In [ ]:
TGraph.Show(g, vertexGroupKey="id", vertexGroups=[0,1,2,3,4], vertexSize=5, vertexLabelKey="id", colorScale="thermal")

In [ ]:
faces = CellComplex.Faces(cc)
selectors = []
for face in faces:
    centroid = Topology.Centroid(_as_topology(face))
    z = Vertex.Z(_as_topology(centroid))
    if z < 6:
        cat = 0
    elif z < 12:
        cat = 1
    elif z < 18:
        cat = 2
    elif z < 24:
        cat = 3
    else:
        cat = 4
    d = Dictionary.ByKeyValue("id", cat)
    centroid = Topology.SetDictionary(centroid, d)
    selectors.append(centroid)

cc = Topology.TransferDictionariesBySelectors(cc, selectors, tranFaces=True)

Topology.Show(cc, faceGroupKey="id", faceGroups=[0,1,2,3,4], faceOpacity=0.5, colorScale="thermal")

In [ ]:
depths = TGraph.DepthMap(g, source=g_vertices[0], key="depthmap")
depthmap = sorted(set(depths.values()))
print(depthmap)

In [ ]:
for i, g_vertex in enumerate(g_vertices):
    dm = depthmap[i]
    d = Dictionary.ByKeyValue("depthmap", dm)
    g_vertex = Topology.SetDictionary(g_vertex, d)

TGraph.Show(g, vertexGroupKey="depthmap", vertexGroups=depthmap, vertexLabelKey="depthmap")

In [ ]:
print(cc)

In [ ]:
edges = CellComplex.Edges(cc)
vertices = CellComplex.Vertices(cc)
new_g = TGraph.ByVerticesEdges(vertices, edges)
TGraph.Show(new_g)